# Easy Paray Painter

This notebook provides a simple interface to interact with the GAN model trained on Laura Paray's seascape and landscape paintings, including the ability to generate new pictures. Be sure to check the readme if you aren't sure how to get started.

## Setup
This section sets up prequisites for the system, which must have already CUDA 10+. The easiest way to run this code without having a prior setup is Google Colab.

In [ ]:
#run this if you need to check your CUDA version
#nvcc --version

#### Important! Google Drive Link
The next cell is a little unusual because it only works on Colab and requires some input from you. Assuming you **are** on Colab, and that you've been following the directions so far, this is a vital step, so I'll explain what's happening and why. Running the cell will output a link to authorize your Google sign-in. Follow the prompt to get a code, copy that code (should be a copy button next to it). Come back here, you should see a small text box in the output of the cell. Paste the code in the box and press enter. What that does is connects your Google Drive to this runtime environment, allowing us to use Google Drive much like we would a regular hard drive.

In [ ]:
#if running on Colab, you can use Google Drive for storage after running this cell
##from google.colab import drive
##drive.mount('/content/drive')

##### Requirements
The next cell writes a requirements.txt file, simply to make it possible to run this notebook in an almost self-contained manner, as long as the model, nlist.db, and factor.pt are accessible. To make the notebook fully self-contained would require the files to be hosted somewhere, along with additional code, and would come with security considerations. Some of the requirements are from earlier experimental versions and aren't required currently, but could be required in the future.

In [ ]:
#local version comes with a different requirements.txt

##### Download and install requirements
The next cell tends to take at least three minutes in Colab

In [ ]:
#install required packages, including pytorch 1.7.1
#if you're on your own PC make sure you know EXACTLY what you are doing here!
!pip install -r requirements.txt

In [ ]:
# depending on the system, you may need to uncomment the next line to install the correct version of pytorch
#!pip install torch==1.7.1+cu110 torchvision==0.8.2+cu110 torchaudio==0.7.2 -f https://download.pytorch.org/whl/torch_stable.html

In [ ]:
#adds open-source GAN modules
#same precaution: if you're on your own PC make sure you know EXACTLY what you are doing here!
!git clone https://github.com/CurtisASmith/stylegan2-pytorch

In [ ]:
# Here we store paths for the model, name data, and factor file. 
# Change the file path as necessary. Probably something like "content/drive/MyDrive/ParayPainter/models/paray-512px.pt" on Colab.
model = "./models/paray-512px.pt"
nlist = "./models/nlist.db"
factor = "./models/factor.pt"
dirname = "./latentexplore/"

In [ ]:
import os
os.chdir("stylegan2-pytorch")

The last part of setup is to define a couple of functions below. The import can take about 45 seconds.

In [ ]:
#define functions for later
import torch
from torchvision import utils
from IPython.display import Image
from model import Generator
from time import localtime, strftime
from tqdm import tqdm
import random
import csv

############# - Returns a list of names the length of listcount. 
# read_db() # - User-specified 'count' is passed via paray_gen(). 
############# - Should never be a reason to change this, but a smarter naming system could replace it (extension idea)

def read_db(listcount):
    names = []
    selected = []
    with open(nlist,'r') as csvfile:
        reader = csv.reader(csvfile,skipinitialspace=True)
        names = list(reader)
    random.shuffle(names)
    for x in range(listcount):
        selected.append(names.pop(x))
    return selected

################ - This runs the experimental vector exploration code. Something extra for fun.
#              # - No truncation control in the interface, can be modified here in the code.
# gen_factor() # - Hard-coded to run the paray-512px.pt model, other models may not be compatible
#              # - savename is a file name concat. from dirname+timestamped image name. factor should be path to factor.pt.
################ - vec and scale set with interface sliders, creates scalar that determines the direction to 'explore' in.

def gen_factor(savename,factor,vec,scale):
    device = "cuda"
    eigvec = torch.load(factor)["eigvec"].to(device)
    ckpt = torch.load(model)
    g = Generator(512, 512, 2, channel_multiplier=2).to(device)
    g.load_state_dict(ckpt["g_ema"], strict=False)

    trunc = g.mean_latent(4096)

    latent = torch.randn(1, 512, device=device)
    latent = g.get_latent(latent)
    scalar = scale * eigvec[:, vec].unsqueeze(0)

    img, _ = g(
            [latent - scalar],
            truncation=0.7,
            truncation_latent=trunc,
            input_is_latent=True,
        )
    utils.save_image(img,savename)

In [ ]:
############### = Should work with no problems now, directions in the interface.
# paray_gen() # = Took some fiddling to make it work nicely with the model.
############### = Anything worth changing is in the interface.

def paray_gen(model,p_num,t_a,t_b,norm=True):
    device = "cuda"
    g_ema = Generator(
        512, 512, 2, channel_multiplier=2
    ).to(device)
    model = torch.load(model)
    truncation = random.uniform(t_a,t_b)
    g_ema.load_state_dict(model["g_ema"])
    
    with torch.no_grad():
        mlatent = g_ema.mean_latent(4096)
        names = read_db(p_num)
        time = "./gen"+strftime("%Y-%d%b-%H%M-%S",localtime())+"/"
        os.mkdir(time)
        g_ema.eval()
        for i in tqdm(range(p_num)):
            zseed = torch.randn(1, 512, device=device)
            truncation = random.uniform(t_a,t_b)
            mlatent = g_ema.mean_latent(4096)
            name = str(names[i]).translate({ord('['): '', ord(']'): '', ord('\''): ''})
            sample, _ = g_ema([zseed], truncation=truncation, truncation_latent=mlatent)
            utils.save_image(
                sample,
                time+name+".png",
                normalize=norm,
                range= (-1, 1),
            )
        print("\nOperation complete. Generated {p_num!s} images and saved them to \"{time!s}\"".format(p_num=p_num, time=time))

With that, setup is complete! The next two cells can be run in any order, indefinitely until you exit the program. Google Colab will automatically end the session if you're idle long enough, but generally gives you hours of free GPU time if you actually use it.

## Usage

In [ ]:
# Experimental controlled generation 
#This cell is an example of how to 'control' image generation.
#The default values will likely generate a picture of waves every time, demonstrating that it is possible to determine controllable variables in the model.
#Try changing vec to 5 and keep the scale around +5 to +10. I expect that generates sunset variations most of the time.
#Experimental, see latent-exploration-notes.txt for ideas if you want to try exploring. I only documented ten vectors, there are hundreds. You'll have to change the maximum for the "vec" slider to go past ten.
#Outputs will be in the folder /content/drive/MyDrive/ParayPainter/latentexplore/ if the path wasn't changed earlier
#Subjective opinion: Some of the individual images from this generator are the best this model is capable of creating.

savename = dirname + strftime("%Y-%d%b-%H%M-%S",localtime()) + ".png"
vec = 1 #<<--<< change those 
scale = 7 #<<--<< values!!

gen_factor(savename,factor,vec,scale)
Image(filename=savename)

In [ ]:
# Primary painting tool
# Finally, this is where the AI can generate paintings in bulk, up to approximately **1950** at a time. See ToC 2.1 note for details. 
#Each image in a folder will have a unique name, generated by GPT-3. 
#t_a and t_b set the truncation range of the generator.
#In general, higher truncation means less variety but higher quality, but this does not always hold true. Unchecking norm on the other hand is certain to make the model more creative at the sacrifice of quality.

#change these values
count = 10 #<<--<< number of images to generate
t_a = 0.5 #<<--<< t_a
t_b = 0.5 #<<--<< t_b
norm = True #<<--<< norm

paray_gen(model,count,t_a,t_b,norm)